# 5G Network Intelligence — Available Supervision Integrity and Replay

Verify the completed joint/available-label experiment independently. This audit never trains or calibrates.

In [ ]:
# Default review does not access checkpoints or run inference.
RUN_AUDIT = False


## Provenance and temporal boundary
Verify all source/input checksums. Keep checkpoints in attached Kaggle Input; output aggregate evidence only. Use original fit/stop/score endpoints 335/383/443 and 395/443/503, context 168h, horizons 1/6/24h. Original hours 504 onward remain redacted.

In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, json, hashlib, shutil
PROJECT = Path.cwd()
if not (PROJECT/'configs').is_dir():
    PROJECT = PROJECT.parent
def verified_input(filename, digest):
    matches = [p for p in Path('/kaggle/input').rglob(filename)
               if hashlib.sha256(p.read_bytes()).hexdigest() == digest]
    if len(matches) != 1:
        raise ValueError('Exactly one pinned input manifest required')
    source = matches[0]
    manifest = json.loads(source.read_text())
    for name, expected in manifest['files'].items():
        relative = PurePosixPath(name)
        if relative.is_absolute() or '..' in relative.parts or ':' in name or '\\' in name:
            raise ValueError('Unsafe input path')
        path = source.parent/name
        if not path.resolve().is_relative_to(source.parent.resolve()) or path.is_symlink():
            raise ValueError('Input path escapes manifest root')
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError('Input checksum differs: '+name)
    return source.parent, manifest


In [ ]:
if RUN_AUDIT:
    assert Path('/kaggle/working').is_dir(), 'Hosted Kaggle only.'
    base_root, base = verified_input('source_manifest.json',
        '3069c57c9f64f809464ead2fae973fbaa35a86c3a46884e15cc693354eecab46')
    code_root, code = verified_input('partial_supervision_source_manifest.json', 'f4c54ca989621a1ef80d89d57573c31ada7a5363e664e91876dc8d65f1dc0dac')
    assert not code['data_or_checkpoint_included']
    PROJECT = Path('/kaggle/working/partial_supervision_audit')
    PROJECT.mkdir(exist_ok=True)
    # Reuse only canonical original panel/split. External source corpora are unused.
    for name in ['data/processed/panel.npz', 'data/processed/panel_meta.json', 'data/processed/split.json']:
        target = PROJECT/name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(base_root/name, target)
    for name in code['files']:
        target = PROJECT/name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(code_root/name, target)
    os.chdir(PROJECT)
    sys.path.insert(0, str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = code['source_commit']
    print('Verified training source:', code['source_commit'])


## Independent acceptance checks
Verify exact artifact coverage, fit-only preprocessing, seed-reconstructed cold weights and bitwise equality across arms. Recount observed labels per horizon, check epoch training/stopping counts and actual optimizer steps. Reconstruct all saved MAE/WAPE from the full population and replay history-only forecasts at fixed atol 0.0001. Require genuine patience stopping; an epoch cap does not pass this audit.

In [ ]:
if RUN_AUDIT:
    import urllib.request, subprocess
    # Retrieve public verifier code by immutable commit and exact content hash.
    url = 'https://raw.githubusercontent.com/trbk07/5g-network-intelligence/82b1dc6e86b74995c932f1a6cc97291d3fb53a67/scripts/audit_partial_supervision_cloud.py'
    with urllib.request.urlopen(url, timeout=60) as response:
        code = response.read()
    assert hashlib.sha256(code).hexdigest() == '37eb42ac30d23ed2e39cc3141f96e59266def93e803303cfbe7ee329d43e8e6f'
    script = PROJECT/'audit_partial_supervision_cloud.py'
    script.write_bytes(code)
    roots = [p.parent for p in Path('/kaggle/input').rglob('manifest.json')
             if p.parent.name == 'partial_supervision'
             and all((p.parent/arm).is_dir() for arm in ['joint', 'available'])]
    assert len(roots) == 1, 'Attach exactly one completed paired experiment'
    subprocess.run([sys.executable, str(script), '--artifacts', str(roots[0]),
        '--source-commit', 'a2d7ff0896d897942770473871289a66c612e8db'], check=True)
else:
    print('Review mode: no fitting or checkpoint access.')


## Metrics and limitations
MAE uses physical DLPRB units; WAPE divides summed absolute error by summed absolute observed load. Natural window counts and optimizer exposure differ between arms, so this is not a pure label-count effect. Reused development does not establish fresh generalization. Integrity PASS alone is not a quality gain or permission to promote a model.

In [ ]:
import pandas as pd
from IPython.display import display
result_path = Path('/kaggle/working/partial_supervision_verified_metrics.json') if RUN_AUDIT else PROJECT/'reports/partial_supervision_verified_metrics.json'
if result_path.exists():
    result = json.loads(result_path.read_text(encoding='utf-8'))
    frame = pd.DataFrame(result['metrics'])
    frame['WAPE (%)'] = 100*frame['wape']
    display(frame)
    display(pd.DataFrame(result['exposure']))
else:
    print('Independent verification pending.')
